In [1]:
# Cell 1: Setup & Project Path
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from delta import configure_spark_with_delta_pip
import time

# สร้าง Spark Session ด้วยค่าคอนฟิกที่ถูกต้องตามคู่มือ Delta Lake
builder = SparkSession.builder \
    .appName("Optimization & Analytics") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

# ใช้ configure_spark_with_delta_pip เพื่อโหลด JARs อัตโนมัติ
spark = configure_spark_with_delta_pip(builder).getOrCreate()

26/02/20 16:32:55 WARN Utils: Your hostname, RATCHANON-PAN.local resolves to a loopback address: 127.0.0.1; using 10.24.10.74 instead (on interface en0)
26/02/20 16:32:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/ratchanon.pan/.ivy2/cache
The jars for the packages stored in: /Users/ratchanon.pan/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ea9014d4-ea1b-4074-a70e-eae3a6b4a180;1.0
	confs: [default]


:: loading settings :: url = jar:file:/Users/ratchanon.pan/ecommerce-analytics/venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 180ms :: artifacts dl 13ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-ea9014d4-ea1b-4074-a70e-eae3a6b4a180
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/7ms)
26/02/20 16:32:5

In [4]:
def get_project_root():
    path = os.path.abspath(os.getcwd())
    while os.path.basename(path) != 'ecommerce-analytics':
        new_path = os.path.dirname(path)
        if new_path == path:
            break
        path = new_path
    return path

def get_data_path(layer, filename):
    project_root = get_project_root()  # ← เรียกใช้ที่นี่
    layer_paths = {
        "bronze": os.path.join(project_root, "data", "bronze", "olist"),
        "silver": os.path.join(project_root, "data", "silver"),
        "gold":   os.path.join(project_root, "data", "gold"),
    }
    if layer not in layer_paths:
        raise ValueError(f"Unknown layer: '{layer}'. Must be one of {list(layer_paths.keys())}.")
    return os.path.join(layer_paths[layer], filename)

In [5]:
df_items = spark.read.csv(get_data_path("bronze", "olist_order_items_dataset.csv"), header=True, inferSchema=True)
df_items.show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

In [6]:
df_orders = spark.read.format("delta").load(get_data_path("silver", "orders"))
df_orders.show(5)

26/02/20 16:47:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+--------------------+------------+------------------------+-------------------------+--------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|order_delivered_timestamp|      ingestion_date|
+--------------------+--------------------+------------+------------------------+-------------------------+--------------------+
|d9eabc69f974b3d08...|d4be0795e8fa7ea79...|   delivered|     2018-03-21 19:32:50|      2018-03-28 18:56:52|2026-02-20 11:41:...|
|c9fbf88f9c58364e0...|7ec5b53960d508118...|   delivered|     2018-02-05 16:47:14|      2018-02-20 19:59:00|2026-02-20 11:41:...|
|fd01a48a7d75383a3...|014f2d069b53eec84...|   delivered|     2017-08-18 14:17:44|      2017-08-28 17:04:11|2026-02-20 11:41:...|
|6d4616de4341417e1...|adc1b0d30fe2b52bd...|   delivered|     2017-04-15 11:46:19|      2017-05-05 15:56:20|2026-02-20 11:41:...|
|c606769bddf9fb8b9...|a4b29e455132615d4...|   delivered|     2017-06-15 13:17:07|      2017-06-23

In [7]:
df_fact_items = df_items \
    .join(df_orders, "order_id") \
    .withColumn("order_date_key", date_format("order_purchase_timestamp", "yyyyMMdd").cast("int")) \
    .select(
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value",
        "order_date_key",
        "order_purchase_timestamp"
    )
df_fact_items.show()

+--------------------+-------------+--------------------+--------------------+------+-------------+--------------+------------------------+
|            order_id|order_item_id|          product_id|           seller_id| price|freight_value|order_date_key|order_purchase_timestamp|
+--------------------+-------------+--------------------+--------------------+------+-------------+--------------+------------------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|  58.9|        13.29|      20170913|     2017-09-13 08:59:02|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...| 239.9|        19.93|      20170426|     2017-04-26 10:53:06|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...| 199.0|        17.87|      20180114|     2018-01-14 14:33:31|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...| 12.99|        12.79|      20180808|     2018-08-08 10:00:35|
|00042b26cf59d7ce6..

In [ ]:
df_fact_items.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("order_date_key") \
    .save(get_data_path("gold", "fact_order_items"))

print("✅ fact_order_items created with date partitioning")

✅ fact_order_items created with date partitioning


In [9]:
# Cell 3: Query BEFORE Z-Ordering (measure performance)
start = time.time()

result = spark.read.format("delta").load(get_data_path("gold", "fact_order_items")) \
    .filter(col("product_id") == "aca2eb7d00ea1a7b8ebd4e68314663af") \
    .groupBy("product_id") \
    .agg(
        sum("price").alias("total_revenue"),
        count("order_id").alias("order_count")
    )

result.show()
before_time = time.time() - start
print(f"Query time BEFORE optimization: {before_time:.2f}s")

+--------------------+-----------------+-----------+
|          product_id|    total_revenue|order_count|
+--------------------+-----------------+-----------+
|aca2eb7d00ea1a7b8...|37608.90000000003|        527|
+--------------------+-----------------+-----------+

Query time BEFORE optimization: 2.66s


In [10]:
# Cell 4: Apply Z-Ordering on product_id
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, get_data_path("gold", "fact_order_items"))
delta_table.optimize().executeZOrderBy("product_id")

print("✅ Z-Ordering applied on product_id")

✅ Z-Ordering applied on product_id


In [11]:
# Cell 5: Query AFTER Z-Ordering
start = time.time()

result = spark.read.format("delta").load(get_data_path("gold", "fact_order_items")) \
    .filter(col("product_id") == "aca2eb7d00ea1a7b8ebd4e68314663af") \
    .groupBy("product_id") \
    .agg(
        sum("price").alias("total_revenue"),
        count("order_id").alias("order_count")
    )

result.show()
after_time = time.time() - start
print(f"Query time AFTER optimization: {after_time:.2f}s")
print(f"Improvement: {((before_time - after_time) / before_time * 100):.1f}%")

+--------------------+-------------+-----------+
|          product_id|total_revenue|order_count|
+--------------------+-------------+-----------+
|aca2eb7d00ea1a7b8...|      37608.9|        527|
+--------------------+-------------+-----------+

Query time AFTER optimization: 1.18s
Improvement: 55.7%


In [ ]:
# Cell 6: Create aggregated Gold table - customer_analytics
df_customers = spark.read.format("delta").load(get_data_path("gold", "dim_customer"))
df_orders = spark.read.format("delta").load(get_data_path("gold", "fact_orders"))
df_items = spark.read.format("delta").load(get_data_path("gold", "fact_order_items"))

print(f"customer")
df_customers.show(5)

customer
+-------------+--------------------+--------------------+------------------------+--------------------+--------------+--------------------+-------------------+----------+
|surrogate_key|customer_natural_key|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|      effective_date|           end_date|is_current|
+-------------+--------------------+--------------------+------------------------+--------------------+--------------+--------------------+-------------------+----------+
|            0|0001fd6190edaaf88...|94b11d37cd61cb299...|                   29830|        nova venecia|            ES|2026-02-20 15:55:...|9999-12-31 00:00:00|      true|
|            1|0004164d20a9e969a...|104bdb7e6a6cdceaa...|                   13272|            valinhos|            SP|2026-02-20 15:55:...|9999-12-31 00:00:00|      true|
|            2|000f17e290c26b285...|74541fbb7526dabec...|                   98400|frederico westphalen|            RS|2026-02-20 15:55:.

In [21]:
customer_analytics = (
    df_orders
    .join(df_customers.filter(col("is_current") == True), 
          df_orders["customer_key"] == df_customers["surrogate_key"])
    .join(df_items.drop("order_purchase_timestamp"), "order_id")  # ← drop ออกก่อน
    .groupBy("customer_natural_key", "customer_city", "customer_state")
    .agg(
        count("order_id").alias("total_orders"),
        sum("price").alias("total_revenue"),
        avg("price").alias("avg_order_value"),
        min("order_purchase_timestamp").alias("first_order_date"),
        max("order_purchase_timestamp").alias("last_order_date")
    )
    .withColumn("customer_lifetime_days", 
                datediff(col("last_order_date"), col("first_order_date")))
)

In [23]:
customer_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .save(get_data_path("gold", "customer_analytics"))
print("✅ customer_analytics aggregation created")

✅ customer_analytics aggregation created


In [ ]:
# Cell 6: Create aggregated Gold table - customer_analytics
df_customers = spark.read.format("delta").load(get_data_path("gold", "dim_customer"))
df_orders = spark.read.format("delta").load(get_data_path("gold", "fact_orders"))
df_items = spark.read.format("delta").load(get_data_path("gold", "fact_order_items"))

customer_analytics = df_orders \
    .join(df_customers.filter(col("is_current") == True), 
          df_orders["customer_key"] == df_customers["surrogate_key"]) \
    .join(df_items, "order_id") \
    .groupBy(
        "customer_natural_key",
        "customer_city",
        "customer_state"
    ) \
    .agg(
        count("order_id").alias("total_orders"),
        sum("price").alias("total_revenue"),
        avg("price").alias("avg_order_value"),
        min("order_purchase_timestamp").alias("first_order_date"),
        max("order_purchase_timestamp").alias("last_order_date")
    ) \
    .withColumn("customer_lifetime_days", 
                datediff(col("last_order_date"), col("first_order_date")))

customer_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .save("../data/gold/customer_analytics")

print("✅ customer_analytics aggregation created")

# Cell 7: Apply Z-Ordering on customer_analytics
delta_analytics = DeltaTable.forPath(spark, "../data/gold/customer_analytics")
delta_analytics.optimize().executeZOrderBy("customer_natural_key")

print("✅ Z-Ordering applied on customer_natural_key")

# Cell 8: Test optimized query
start = time.time()

top_customers = spark.read.format("delta").load("../data/gold/customer_analytics") \
    .orderBy(col("total_revenue").desc()) \
    .limit(10)

top_customers.show(truncate=False)
query_time = time.time() - start

print(f"\nTop customers query time: {query_time:.2f}s")

# Cell 9: Create product analytics
product_analytics = df_items \
    .groupBy("product_id") \
    .agg(
        count("order_id").alias("total_orders"),
        sum("price").alias("total_revenue"),
        avg("price").alias("avg_price"),
        sum("freight_value").alias("total_freight")
    ) \
    .orderBy(col("total_revenue").desc())

product_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .save("../data/gold/product_analytics")

print("✅ product_analytics created")

# Cell 10: Summary stats
print("\n=== Gold Layer Summary ===")
print(f"dim_customer: {spark.read.format('delta').load('../data/gold/dim_customer').count()} records")
print(f"dim_product: {spark.read.format('delta').load('../data/gold/dim_product').count()} records")
print(f"fact_orders: {spark.read.format('delta').load('../data/gold/fact_orders').count()} records")
print(f"fact_order_items: {spark.read.format('delta').load('../data/gold/fact_order_items').count()} records")
print(f"customer_analytics: {spark.read.format('delta').load('../data/gold/customer_analytics').count()} records")
print(f"product_analytics: {spark.read.format('delta').load('../data/gold/product_analytics').count()} records")